In [1]:
#run install for gym and biped walker env.

%pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:

%brew install swig

UsageError: Line magic function `%brew` not found.


In [ ]:
%swig -version
%brew -version

%pip install gymnasium stable-baselines3 



In [6]:
tensorboard --logdir logs/

SyntaxError: invalid syntax (266949774.py, line 1)

In [5]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os

def train():
    # Create a monitored environment for logging
    log_dir = "logs/"
    os.makedirs(log_dir, exist_ok=True)
    env = make_vec_env(
        'BipedalWalker-v3',
        n_envs=8,
        env_kwargs={"render_mode": None},
        monitor_dir=log_dir
    )

    # Define the PPO model with hyperparameters
    model = PPO(
        "MlpPolicy",
        env,
        verbose=2,
        n_epochs=15,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
    )

    # Add callbacks for evaluation and checkpointing
    eval_env = Monitor(gym.make('BipedalWalker-v3', render_mode=None))
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path="./logs/best_model",
        log_path="./logs/eval",
        eval_freq=10000,
        deterministic=True,
        render=False,
    )
    checkpoint_callback = CheckpointCallback(
        save_freq=50000,
        save_path="./logs/checkpoints",
        name_prefix="ppo_bipedalwalker"
    )

    # Train the model
    model.learn(total_timesteps=2_000_000, callback=[eval_callback, checkpoint_callback])

    # Save the final model
    model.save("ppo_bipedalwalker")
    env.close()

if __name__ == "__main__":
    train()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 428      |
|    ep_rew_mean     | -111     |
| time/              |          |
|    fps             | 5841     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 505          |
|    ep_rew_mean          | -110         |
| time/                   |              |
|    fps                  | 3818         |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0058019273 |
|    clip_fraction        | 0.0611       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.7         |
|    explained_variance   | -0.000296    

In [4]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import pandas as pd

def train():
    env = make_vec_env('BipedalWalker-v3', n_envs=8,
                       env_kwargs={"render_mode": None})
    model = PPO(
        "MlpPolicy",
        env,
        verbose=2,
        n_epochs = 15,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
    )

    model.learn(total_timesteps=2_000_000)

    model.save("ppo_bipedalwalker")
    env.close()

if __name__ == "__main__":
    train()
    import matplotlib.pyplot as plt

    # Load the training log
    log_dir = "logs/"
    monitor_file = log_dir + "monitor.csv"

    # Read the monitor file
    data = pd.read_csv(monitor_file, skiprows=1)
    data['cumulative_timesteps'] = data['l'].cumsum()

    # Plot the rewards
    plt.figure(figsize=(10, 6))
    plt.plot(data['cumulative_timesteps'], data['r'], label='Episode Reward')
    plt.xlabel('Timesteps')
    plt.ylabel('Reward')
    plt.title('Training Progress')
    plt.legend()
    plt.grid()
    plt.show()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 324      |
|    ep_rew_mean     | -109     |
| time/              |          |
|    fps             | 6492     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 16384    |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 346          |
|    ep_rew_mean          | -112         |
| time/                   |              |
|    fps                  | 3919         |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 32768        |
| train/                  |              |
|    approx_kl            | 0.0069251214 |
|    clip_fraction        | 0.0642       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.73        |
|    explained_variance   | 0.00101      

FileNotFoundError: [Errno 2] No such file or directory: 'logs/monitor.csv'

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

def run():
    env = gym.make("BipedalWalker-v3", render_mode="human")
    model = PPO.load("ppo_bipedalwalker")

    obs, _ = env.reset()

    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)

        if terminated or truncated:
            obs, _ = env.reset()

if __name__ == "__main__":
    run()

KeyboardInterrupt: 